In [6]:
import pandas as pd

In [7]:
def map_product(product):

    if product in [
        "Credit card",
        "Credit card or prepaid card"
    ]:
        return "Credit Card"

    elif product == "Checking or savings account":
        return "Savings Account"

    elif product in [
        "Money transfer, virtual currency, or money service",
        "Money transfers"
    ]:
        return "Money Transfer"

    else:
        return "Personal Loan"

In [8]:
df = pd.read_csv('../data/filtered_complaints.csv')

df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID',
       'narrative_length', 'cleaned_narrative'],
      dtype='str')

In [9]:
df["product_category"] = (
    df["Product"]
    .apply(map_product)
)
df.columns


Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID',
       'narrative_length', 'cleaned_narrative', 'product_category'],
      dtype='str')

In [10]:
df["product_category"].value_counts()

product_category
Credit Card        189334
Savings Account    140319
Money Transfer      98685
Personal Loan       35595
Name: count, dtype: int64

In [11]:
df["product_category"].value_counts(normalize=True)

product_category
Credit Card        0.408106
Savings Account    0.302455
Money Transfer     0.212714
Personal Loan      0.076724
Name: proportion, dtype: float64

In [12]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    df,
    train_size=12000,
    stratify=df["product_category"],
    random_state=42
)

In [13]:
sample_df["product_category"].value_counts(normalize=True)

product_category
Credit Card        0.408083
Savings Account    0.302417
Money Transfer     0.212750
Personal Loan      0.076750
Name: proportion, dtype: float64

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

generate chunks


In [15]:
texts = []

for text in sample_df["cleaned_narrative"]:

    chunks = splitter.split_text(text)

    texts.extend(chunks)

creating metadata

In [16]:
documents = []

for idx, row in sample_df.iterrows():

    chunks = splitter.split_text(
        row["cleaned_narrative"]
    )

    for chunk in chunks:

        documents.append(
            {
                "text": chunk,
                "complaint_id": idx,
                "product_category":
                    row["product_category"]
            }
        )

In [17]:
from sentence_transformers import (
    SentenceTransformer
)

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\mijuu\Documents\mijuuhailu\intelligent-complaint-analysis\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mijuu\Documents\mijuuhailu\intelligent-complaint-analysis\myenv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mijuu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need

Generate Embeddings

In [18]:
texts = [
    doc["text"]
    for doc in documents
]

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches: 100%|██████████| 1102/1102 [09:59<00:00,  1.84it/s]


building index

In [21]:
import faiss
import numpy as np

embeddings = np.array(
    embeddings,
    dtype="float32"
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(embeddings)

saving

In [26]:
import os

os.makedirs("../data/vector_store", exist_ok=True)

faiss.write_index(
    index,
    "../data/vector_store/faiss_index.bin"
)

In [28]:
import pickle

with open(
    "../data/vector_store/metadata.pkl",
    "wb"
) as f:

    pickle.dump(documents, f)

In [29]:
print(len(documents))
print(index.ntotal)

35234
35234
